In [1]:
import os
import sys
import glob
import pandas as pd

# --- 1. Path Setup ---
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
root_dir = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

config_dir = os.path.join(root_dir, "config")
raw_data_dir = os.path.join(root_dir, "data", "raw")
processed_data_dir = os.path.join(root_dir, "data", "processed")
os.makedirs(processed_data_dir, exist_ok=True)

master_output_file = os.path.join(processed_data_dir, "master_ontology_dataset.csv")

# --- 2. Load Schema & Registry ---
sys.path.append(config_dir)
try:
    from ingest_schema_manager import COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py.")

registry_file = os.path.join(config_dir, "source_registry.csv")
expected_prefixes = []
if os.path.exists(registry_file):
    expected_prefixes = pd.read_csv(registry_file)['Prefix'].str.lower().tolist()

# --- 3. Dynamic File Discovery & Registry Cross-Check ---
raw_files = glob.glob(os.path.join(raw_data_dir, "raw_*.csv"))
found_prefixes = [os.path.basename(f).replace("raw_", "").replace(".csv", "").lower() for f in raw_files]

print("--- 1. Pipeline Status ---")
for expected in expected_prefixes:
    if expected not in found_prefixes:
        print(f"[!] MISSING DATA: '{expected}' is in the registry but has no raw CSV file.")
        
if not raw_files:
    raise FileNotFoundError(f"No raw files found in {raw_data_dir}. Halting consolidation.")

# --- 4. Strict QA & Consolidation ---
print(f"\n--- 2. Validating {len(raw_files)} Raw Files ---")
dataframes = []
total_raw_rows = 0

for file_path in raw_files:
    filename = os.path.basename(file_path)
    try:
        df = pd.read_csv(file_path, dtype=str) # Read as strings to prevent Pandas type guessing
        total_raw_rows += len(df)
        
        # QA Check 1: Extra Columns (Data Loss Warning)
        extra_cols = set(df.columns) - set(COLUMN_ORDER)
        if extra_cols:
            print(f"[!] WARNING - EXTRA COLUMNS in {filename}: {extra_cols}")
            print(f"    -> Action: These columns will be DROPPED from the master file.")
            
        # QA Check 2: Missing Columns (Structure Warning)
        missing_cols = set(COLUMN_ORDER) - set(df.columns)
        if missing_cols:
            print(f"[!] WARNING - MISSING COLUMNS in {filename}: {missing_cols}")
            print(f"    -> Action: These will be injected as blank strings.")
            for col in missing_cols:
                df[col] = ""
        
        if not extra_cols and not missing_cols:
            print(f"[PASS] {filename} perfectly matches the 14-column schema.")

        # Filter strictly to our schema
        dataframes.append(df[COLUMN_ORDER])
        
    except pd.errors.EmptyDataError:
        print(f"[!] WARNING - EMPTY FILE: {filename}. Skipping.")
    except Exception as e:
        print(f"[!] ERROR reading {filename}: {e}")

# --- 5. Merge & Deduplicate ---
print("\n--- 3. Consolidation & Deduplication ---")
if dataframes:
    master_df = pd.concat(dataframes, ignore_index=True)
    
    # Calculate duplicates based strictly on URI
    duplicate_mask = master_df.duplicated(subset=['URI'], keep='last')
    duplicate_count = duplicate_mask.sum()
    
    if duplicate_count > 0:
        print(f"Identified {duplicate_count} duplicate URIs across the raw files.")
        print("    -> Action: Retaining the last ingested version of each URI.")
        master_df = master_df[~duplicate_mask]
        
    final_count = len(master_df)
    
    # --- 6. Export ---
    master_df.to_csv(master_output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n--- 4. Final Report ---")
    print(f"Total Rows Ingested:   {total_raw_rows}")
    print(f"Duplicates Removed:   -{duplicate_count}")
    print(f"Final Master Rows:     {final_count}")
    print(f"\nSUCCESS: Golden dataset saved to {master_output_file}")

--- 1. Pipeline Status ---
[!] MISSING DATA: 'hl7v3' is in the registry but has no raw CSV file.
[!] MISSING DATA: 'hl7v2' is in the registry but has no raw CSV file.

--- 2. Validating 5 Raw Files ---
[PASS] raw_elsst.csv perfectly matches the 14-column schema.
[PASS] raw_hl7.csv perfectly matches the 14-column schema.
[PASS] raw_lcdgt.csv perfectly matches the 14-column schema.
[PASS] raw_loinc.csv perfectly matches the 14-column schema.
[PASS] raw_snomed.csv perfectly matches the 14-column schema.

--- 3. Consolidation & Deduplication ---

--- 4. Final Report ---
Total Rows Ingested:   841
Duplicates Removed:   -0
Final Master Rows:     841

SUCCESS: Golden dataset saved to c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\processed\master_ontology_dataset.csv
